# Bibliotecas

In [ ]:
#########################
##      PSYCOPG2       ##
#########################
import psycopg2
from psycopg2 import sql
from psycopg2.extras import execute_values


#########################
#    TOMLLIB            #
#########################
import tomllib


#########################
##       PATHLIB       ##
#########################
import pathlib
from pathlib import Path


#########################
##      DATETIME       ##
#########################
import datetime
from datetime import date


#########################
##      pandas         ##
#########################
import pandas as pd


#########################
##      TRACEBACK      ##
#########################
import traceback

# def raiz_do_projeto

In [ ]:
def localizar_raiz_projeto():

    pasta_atual = Path.cwd().resolve()

    for pasta in [
        pasta_atual,
        *pasta_atual.parents
    ]:

        if (
            (pasta / "requirements.txt").exists()
            and
            (pasta / "src").exists()
        ):
            return pasta

    raise FileNotFoundError(
        "Não foi possível localizar a raiz do projeto."
    )

PROJETO_RH = localizar_raiz_projeto()

CAMINHO_CONFIG = (
    PROJETO_RH /
    "config.toml"
)

print(
    f"Raiz do projeto: {PROJETO_RH}"
)

print(
    f"Config encontrado: {CAMINHO_CONFIG.exists()}"
)

# def conexao_db

In [ ]:
def conexao_db():
    conn = None
    cursor = None

    try:

        with open(
            CAMINHO_CONFIG,
            "rb"
            ) as arquivo:

            conn = tomllib.load(arquivo)

            db = conn["risoluto"]

        conn = psycopg2.connect(
            host = db["host"],
            port = db["port"],
            dbname = db["dbname"],
            user = db["user"],
            password = db["password"]
        )

        cursor = conn.cursor()

        return conn, cursor
    
    except Exception as erro:

        print(f"Erro ao conectar com o banco de dados: {erro}")

        return None, None

# def tratamento_de_datas

In [ ]:
def tratamento_de_datas(valor):

    if valor is None:
        return None
    
    if pd.isna(valor):
        return None
    
    if isinstance(valor, datetime):
        return valor.date()
    
    if isinstance(valor,date):
        return valor
    
    valor = str(valor).strip()

    if valor == "":
        return None
    
    formatos = [
        "%d/%m/%Y",
        "%d-%m-%Y",
        "%Y-%m-%d",
        "%Y/%m/%d"
    ]

    for formato in formatos:
        try:
            return datetime.strptime(valor, formato).date()
        except ValueError:
            continue

    return None

# def tratar_inteiro

In [ ]:
def tratar_inteiro(valor):
    if pd.isna(valor):
        return None
    
    valor = str(valor).strip()

    if valor == "":
        return None
    
    try:
        return int(float(valor))
    except (ValueError, TypeError):
        return None

# def tratar_valor_id

In [ ]:
def tratar_valor_id(valor):
        if pd.isna(valor):
                return None

        valor = str(valor).strip()

        if valor == "":
                return None

        return valor

# def atualizar_upsert

In [ ]:
def atualizar_upsert(
        tabela,
        coluna,
        valor,
        conn,
        cursor
):
    
    if valor is None:
        return None
    
    query = sql.SQL("""
            INSERT INTO {tabela} ({coluna})
            VALUES (%s)
            ON CONFLICT ({coluna})
            DO UPDATE SET {coluna} = EXCLUDED.{coluna}
            RETURNING id
    """).format(
        tabela = sql.Identifier(tabela),
        coluna = sql.Identifier(coluna)
    )

    cursor.execute(query, (valor,))

    resultado = cursor.fetchone()

    if resultado:
        return resultado[0]
    
    return None

# def obter_sql_servidor

In [ ]:
def obter_sql_servidor():

    sql_dados_servidor = """
    INSERT INTO tb_fp_por_profissional(
    matricula_data,
    competencia,
    projecao_mensal,
    horas_trabalhadas_competencia,
    dias_faltas,
    horas_faltas,
    horas_dicionais,
    saldo_competencia,
    horas_aprovadas,
    trab_sobre_aviso,
    sobre-aviso,
    horas_plantao,
    unidade,
    matricula,
    especialidade,
    data,
    entrada_1,
    saida_1,
    entrada_2,
    saida_2,
    entrada_3,
    saida_3,
    entrada_4,
    saida_4,
    nao_aprovadas,
    horas_trabalhadas_dia,
    motivo_falta,
    saldo_dia,
    ultima_atualizacao
    )
    VALUES %s
    ON CONFLICT (matricula_data)
    DO UPDATE SET
        competencia = EXCLUDED.competencia,
        projecao_mensal = EXCLUDED.projecao_mensal,
        horas_trabalhadas_competencia = EXCLUDED.horas_trabalhadas_competencia,
        dias_faltas = EXCLUDED.dias_faltas,
        horas_faltas = EXCLUDED.horas_faltas,
        horas_adicionais = EXCLUDED.horas_adicionais,
        saldo_competencia = EXCLUDED.saldo_competencia,
        horas_aprovadas = EXCLUDED.horas_aprovadas,
        trab_sobre_aviso = EXCLUDED.trab_sobre_aviso,
        sobre_aviso = EXCLUDED.sobre_aviso,
        horas_plantao = EXCLUDED.horas_plantao,
        unidade = EXCLUDED.unidade,
        matricula = EXCLUDED.matricula,
        especialidade = EXCLUDED.especialidade,
        data = EXCLUDED.data,
        entrada_1 = EXCLUDED.entrada_1,
        saida_1 = EXCLUDED.saida_1,
        entrada_2 = EXCLUDED.entrada_2,
        saida_2 = EXCLUDED.saida_2,
        entrada_3 = EXCLUDED.entrada_3,
        saida_3 = EXCLUDED.saida_3,
        entrada_4 = EXCLUDED.entrada_4,
        saida_4 = EXCLUDED.saida_4,
        nao_aprovadas = EXCLUDED.nao_aprovadas,
        horas_trabalhadas_dia = EXCLUDED.horas_trabalhadas_dia,
        motivo_falta = EXCLUDED.motivo_falta,
        saldo_dia = EXCLUDED.saldo_dia,
        ultima_atualizacao = EXCLUDED.ultima_atualizacao
    """

# def obter_ids

In [ ]:
def obter_ids(linha, conn, cursor):

    id_competencia = atualizar_upsert(
        tabela = "pk_competencia",
        coluna = "competencia",
        valor = linha["competencia"],
        conn = conn,
        cursor = cursor
    )

    id_especialiade = atualizar_upsert(
        tabela = "pk_especialiade",
        coluna = "especialidade",
        valor = linha["especialidade"],
        conn = conn,
        cursor = cursor
    )

    id_motivo_falta = atualizar_upsert(
        tabela = "pk_motivo_falta",
        coluna = "motivo_falta",
        valor = linha["motivo_falta"],
        conn = conn,
        cursor = cursor
    )

    id_tipo_escala = atualizar_upsert(
        tabela = "pk_tipo_escala",
        coluna = "tipo_escala",
        valor = linha["tipo_escala"],
        conn = conn,
        cursor = cursor
    )

    id_unidade = atualizar_upsert(
        tabela = "pk_unidade",
        coluna = "unidade",
        valor = linha["unidade"],
        conn = conn,
        cursor = cursor
    )

    id_usuario = atualizar_upsert(
        tabela = "",
        coluna = "",
        valor = linha[""],
        conn = conn,
        cursor = cursor
    )